In [14]:
import os
import re
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Configuration
# ============================================================

BASE_DIR = "/scratch/elena/9Li/results"
FV_TAG = "FV_1"

RUNS_BY_BEAM = {
    "-270 MeV/c": [2407, 2408, 2409, 2432],
    "-350 MeV/c": [2374, 2379],
}

# Cylindrical selection used for the vertex plots.
R_CYLINDER_CM = 153.7963
H_CYLINDER_CM = 271.4235

# Final FV box used by merge_and_fv_cut.py, in cm.
FV_BOX = {
    "x": (-20.0, 20.0),
    "y": (-20.0, 20.0),
    "z": (-130.0, 0.0),
}

RUNS = [
    (beam, run)
    for beam, runs in RUNS_BY_BEAM.items()
    for run in runs
]

print(f"Runs to inspect: {len(RUNS)}")
for beam, run in RUNS:
    print(f"  {beam:12s} | Run {run}")

Runs to inspect: 6
  -270 MeV/c   | Run 2407
  -270 MeV/c   | Run 2408
  -270 MeV/c   | Run 2409
  -270 MeV/c   | Run 2432
  -350 MeV/c   | Run 2374
  -350 MeV/c   | Run 2379


In [15]:
# ============================================================
# Exact file matching
# ============================================================

def get_stage_files(processed_dir, stage, sample):
    """
    Return only the intended files for one stage and sample.
    This avoids counting derived files as Stage 1 candidates.
    """
    is_bkg = sample == "bkg"

    if stage == "stage1":
        pattern = os.path.join(processed_dir, "Li9_clusters_chunk_*.pkl")
        regex = (
            r"Li9_clusters_chunk_\d+_BKG\.pkl"
            if is_bkg
            else r"Li9_clusters_chunk_\d+\.pkl"
        )

    elif stage == "multilat":
        pattern = os.path.join(processed_dir, "Li9_clusters_chunk_*_multilat.pkl")
        regex = (
            r"Li9_clusters_chunk_\d+_BKG_multilat\.pkl"
            if is_bkg
            else r"Li9_clusters_chunk_\d+_multilat\.pkl"
        )

    elif stage == "refine":
        pattern = os.path.join(processed_dir, "Refine_Li9_clusters_chunk_*.pkl")
        regex = (
            r"Refine_Li9_clusters_chunk_\d+_BKG\.pkl"
            if is_bkg
            else r"Refine_Li9_clusters_chunk_\d+\.pkl"
        )

    else:
        raise ValueError(f"Unknown stage: {stage}")

    return sorted(
        f for f in glob(pattern)
        if re.fullmatch(regex, os.path.basename(f))
    )


def read_pickle_files(files):
    """Read and concatenate pickle files, preserving an empty result if needed."""
    frames = []

    for path in files:
        try:
            df = pd.read_pickle(path)
            if not isinstance(df, pd.DataFrame):
                print(f"WARNING: not a DataFrame: {path}")
                continue
            frames.append(df)
        except Exception as exc:
            print(f"ERROR reading {path}: {exc}")

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


# ============================================================
# Load all stages
# ============================================================

data = {}
inventory_rows = []

for beam, run in RUNS:
    processed_dir = os.path.join(BASE_DIR, f"run{run}", "processed")
    fv_dir = os.path.join(processed_dir, FV_TAG)

    for sample in ("signal", "bkg"):
        stage_files = {
            stage: get_stage_files(processed_dir, stage, sample)
            for stage in ("stage1", "multilat", "refine")
        }

        final_name = (
            f"Final_FV_Li9_clusters_run{run}_BKG.pkl"
            if sample == "bkg"
            else f"Final_FV_Li9_clusters_run{run}.pkl"
        )
        final_path = os.path.join(fv_dir, final_name)

        dfs = {
            stage: read_pickle_files(files)
            for stage, files in stage_files.items()
        }
        dfs["final"] = (
            pd.read_pickle(final_path)
            if os.path.isfile(final_path)
            else pd.DataFrame()
        )

        data[(run, sample)] = {
            "beam": beam,
            "processed_dir": processed_dir,
            "fv_dir": fv_dir,
            "files": stage_files,
            "final_path": final_path,
            "dfs": dfs,
        }

        for stage in ("stage1", "multilat", "refine", "final"):
            df = dfs[stage]
            n_files = (
                len(stage_files[stage])
                if stage in stage_files
                else int(os.path.isfile(final_path))
            )

            inventory_rows.append({
                "beam": beam,
                "run": run,
                "sample": sample,
                "stage": stage,
                "files": n_files,
                "rows": len(df),
            })

inventory = pd.DataFrame(inventory_rows)

print("FILE AND ROW INVENTORY")
display(
    inventory.pivot_table(
        index=["beam", "run", "sample"],
        columns="stage",
        values=["files", "rows"],
        aggfunc="first",
        fill_value=0,
    )
)

FILE AND ROW INVENTORY


files                         rows                   \
stage                  final multilat refine stage1 final multilat  refine   
beam       run  sample                                                       
-270 MeV/c 2407 bkg        0        0      0      0     0        0       0   
                signal     1        5      5      5     0   110781  110781   
           2408 bkg        0        0      0      0     0        0       0   
                signal     1        6      6      6     0    87568   87568   
           2409 bkg        0        0      0      0     0        0       0   
                signal     1        8      8      8     0    61887   61887   
           2432 bkg        0        0      0      0     0        0       0   
                signal     1        3      3      3     0    66214   66214   
-350 MeV/c 2374 bkg        0        0      0      0     0        0       0   
                signal     1        7      7      7     0   273487  273487   
           2379 bkg        0        0      0      0     0        0       0   
                signal     1       26     26     26     0   194038  194038   

                                
stage                   stage1  
beam       run  sample          
-270 MeV/c 2407 bkg          0  
                signal  110781  
           2408 bkg          0  
                signal   87568  
           2409 bkg          0  
                signal   61887  
           2432 bkg          0  
                signal   66214  
-350 MeV/c 2374 bkg          0  
                signal  273487  
           2379 bkg          0  
                signal  194038